### Prior Setting Requirements
1. 5m로 Rasterize된 임상도 (활용 소프트웨어: ArcGIS - <Feature to Raster>, Reference raster: DEM )
2. 생성된 모든 레스터 파일은 DEM을 기준으로 metedata 동일하게 설정하기 (coord, resolution, extent 등등)

In [ ]:
# Data anaylsis libraries
import sys, site
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from glob import glob
import joblib
from tqdm.notebook import tqdm
import matplotlib.font_manager as fm
# Spatial data libraries
import geopandas as gpd
import scipy.stats as stats
import rasterio
import fiona
# Utiliy for the efficiency improvement
import cupy as cp
from rasterio.windows import Window
from collections import defaultdict
from datetime import datetime
import sys, traceback
from rasterio.transform import xy
from rasterio.windows import transform
%matplotlib inline

try:
    import pyproj
    from pyproj import CRS
except ImportError as e:
    print(e)
    usr_site = site.getusersitepackages()
    if usr_site in sys.path:
        sys.path.remove(usr_site)   # stop picking up pip user packages
        print("Removed user site:", usr_site)
    
    # Now import safely
    import pyproj
    from pyproj import CRS
    print("pyproj OK") 

In [ ]:
# 한글폰트 설정
import matplotlib.font_manager as fm
import matplotlib as mpl

# 예: 나눔고딕 또는 맑은고딕 설정
font_path = "C:/Windows/Fonts/malgun.ttf"  # Windows: 맑은고딕
# font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"  # Linux

font_name = fm.FontProperties(fname=font_path).get_name()
mpl.rc('font', family=font_name)

# 한글 깨짐 방지 (마이너스 부호 처리)
mpl.rcParams['axes.unicode_minus'] = False

### Improvements
1. distribution말고 average 값을 일괄적으로 적용하는 옵션도 만들까?

# Build Distributions of DBH, Height, Crown Density

In [ ]:
# NFI 데이터로 DBH, 수고, 수관밀도 분포 추정 및 추출 (전체 수종에 대한 분포)
# 0-50: gamma distribution or log-normal distribtuijon

class buildDistribution:
    def __init__(self):
        self.attribute = None
        self.results = {}
        self.best_fit = None
        self.data = None
        self.distributions = {"Gamma": stats.gamma,
                              "Log-Normal": stats.lognorm,
                              "Beta": stats.beta,
                              "Weibull": stats.weibull_min,
                              "Exponential": stats.expon,
                              "GEV" : stats.genextreme
                             }
        self.samples = None
        
    def find_best_fit_distribution(self, data, lower, upper, attribute, distributions=None):
        """Finds the best-fitting distribution using the KS test and returns results."""
        self.attribute = attribute
        condition = (data[attribute] > lower) & (data[attribute] <= upper)
        data_filtered = data.loc[condition,attribute].dropna()
        self.data = data_filtered
        if distributions is None:
            distributions = self.distributions
            
        x = np.linspace(data_filtered.min(), data_filtered.max(), 1000)
    
        for name, dist in distributions.items():
            try:
                # Fit distribution to data
                params = dist.fit(data_filtered)
                pdf_fitted = dist.pdf(x, *params)
                # KS(Kolmogorov-Smirnov) test to compare a distribution based on data to a reference probability distribution
                ks_stat, ks_pval = stats.kstest(data_filtered, lambda x: dist.cdf(x, *params))
    
                # Store results
                self.results[name] = {
                    "params": params,
                    "KS Statistic": ks_stat,
                    "p-value": ks_pval,
                    "pdf": pdf_fitted
                }
            except Exception as e:
                print(f"Skipping {name} due to error: {e}")
    
        # Select best fit (highest p-value)
        self.best_fit = max(self.results, key=lambda d: self.results[d]["p-value"])
        
        return self.best_fit, self.results

    def plot_distribution(self):
        """Plot histogram and best fit."""
        x = np.linspace(self.data.min(), self.data.max(), len(self.results[self.best_fit]["pdf"]))
        plt.figure(figsize=(8, 5))
        plt.hist(self.data, bins=30, density=True, color='gray', alpha=0.6, label="Data Histogram")
        plt.plot(x, self.results[self.best_fit]["pdf"], label=f"Best Fit: {self.best_fit}", linewidth=2, color="red")
        plt.xlabel(f"{self.attribute}")
        plt.ylabel("Probability Density")
        plt.title(f"Best-Fitting Probability Distribution for {self.attribute}")
        plt.legend()
        plt.show()
        
        # Print best fit results
        print(f"Best-Fitting Distribution: {self.best_fit}")
        print(f"Parameters: {self.results[self.best_fit]['params']}")
        print(f"KS Statistic: {self.results[self.best_fit]['KS Statistic']}")
        print(f"P-Value: {self.results[self.best_fit]['p-value']}")
        
    def create_samples(self, attribute_name, min_bound=None, max_bound=None, sample_size=1000000, SEED=100):
        rng = np.random.default_rng(SEED)
        best_fit_name = self.best_fit
        params = self.results[best_fit_name]['params']
        best_dist = self.distributions[best_fit_name
                                      ]
        if best_fit_name in ['Gamma', 'Log-Normal', 'Weibull', 'GEV']:
            shape, loc, scale = params
            samples = best_dist.rvs(shape, loc=loc, scale=scale, size=sample_size, random_state=rng)
        if best_fit_name == 'Beta':
            a, b, loc, scale = params
            samples = best_dist.rvs(a, b, loc=loc, scale=scale, size=sample_size, random_state=rng)
        if best_fit_name == 'Eponential':
            loc, scale = params
            samples = best_dist.rvs(loc=loc, scale=scale, size=sample_size, random_state=rng)
        
        if attribute_name == 'DNST_CD': samples[(samples > 0.) & (samples <= 100.)]
        else: samples[(samples > 0.)]
        
        min_bound = min_bound if min_bound != None else min(samples)
        max_bound = max_bound if max_bound != None else max(samples)
        
        return np.clip(samples, min_bound, max_bound)

In [ ]:
# load NFI data
nfi_file = os.path.join(data_dir, r"NFI6-7_cleaned2.csv")
nfi_data_cleaned = pd.read_csv(nfi_file, encoding='cp949')

# NFI 원본
nfi_file2 = os.path.join(data_dir, r"NFI-Integrated(6-7)-Immok-Filtered.xlsx")
if 'NFI6-7' in pd.ExcelFile(nfi_file2).sheet_names:
    nfi_data = pd.read_excel(nfi_file2, sheet_name='NFI6-7')
nfi_data.info()

In [ ]:
# 1회만 시행
# nfi_data['수고'] = nfi_data['수고'] / 100 # m로 변환
# nfi_data['평균수관밀도(%)'] = nfi_data['평균수관밀도(%)'] / 100 # decimal 로 변환

In [ ]:
# 평균 수관밀도
# 속성: ['DNST_CD', 'DMCLS_CD', 'HEIGHT']
SEED = 400
cd_dist = buildDistribution()
attribute = '평균수관밀도(%)'
lower = 0.
upper = 1.0
cd_best_fit, cd_results = cd_dist.find_best_fit_distribution(nfi_data, lower, upper, attribute)
cd_samples = cd_dist.create_samples(attribute_name=attribute, min_bound=lower, max_bound=upper, sample_size=1000000, SEED=SEED)

In [ ]:
min(cd_samples), max(cd_samples)

In [ ]:
# 수고
h_dist = buildDistribution()
attribute = '수고'
lower = 0
upper = 40
h_best_fit, h_results = h_dist.find_best_fit_distribution(nfi_data, lower, upper, attribute)
h_samples = h_dist.create_samples(attribute_name=attribute, min_bound=lower, max_bound=upper, sample_size=1000000, SEED=SEED)

In [ ]:
min(h_samples), max(h_samples)

In [ ]:
h_dist.plot_distribution()

In [ ]:
# DBH(inch)
dbh_dist = buildDistribution()
attribute = '흉고직경'
lower = 0
upper = 110
dbh_best_fit, dbh_results = dbh_dist.find_best_fit_distribution(nfi_data, lower, upper, attribute)
dbh_samples = dbh_dist.create_samples(attribute_name=attribute, min_bound = lower, max_bound = upper, sample_size=1000000, SEED=1000)

In [ ]:
min(dbh_samples), max(dbh_samples)

In [ ]:
dbh_dist.plot_distribution()

In [ ]:
# Visualize the distirbution of the Samples
def visualize_multiple_distribution(data_list, num_of_col):
    num_of_graphs = len(data_list)
    num_of_row = int(np.ceil(num_of_graphs / num_of_col))

    fig, axes = plt.subplots(num_of_row, num_of_col, figsize=(5 * num_of_col, 4))
    axes = axes.flatten()

    # Plot histograms
    for i, data in enumerate(data_list):
        axes[i].hist(data, bins=100)
        axes[i].set_title(f'Distribution {i+1}')

    # Remove empty axes
    for j in range(len(data_list), len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()  
    
data_list = [cd_samples, dbh_samples, h_samples]
num_of_col = 3
visualize_multiple_distribution(data_list, num_of_col)

In [ ]:
# 임상도 파일 읽어오기
gdb_dir = r"H:\CBH\Imsang_merge.gdb"
# gdb에 레이어가 존재한다면 첫번째 레이어 읽어오기
if fiona.listlayers(gdb_dir):
    layer = fiona.listlayers(gdb_dir)
    imsang = gpd.read_file(os.path.join(gdb_dir), layer=layer[0])

In [ ]:
# DBH, 수고, 수관비율 값만 저장
imsang[['DBH(cm)', 'Height(m)', 'CD(%)']] = np.zeros((len(imsang), 3))

 # 1번만 실행
# imsang.reset_index(names='ID', inplace=True)
# imsang['ID'] = imsang['ID'] + 1

# 산림지만 추출하기
# no_imsang_code = [str(i) for i in [78, 81, 82, 83, 91, 92, 93, 94, 95, 99]]
# filtered_imsang = imsang[~imsang['KOFTR_GROU'].isin(no_imsang_code)]
print("추출한 임상도 크기: ", len(filtered_imsang.index), len(imsang.index))
# HEIGHT 코드 정돈
imsang.loc[:, 'HEIGHT'] = imsang['HEIGHT'].apply(lambda x: '40' if x == '42' else x)
imsang.loc[:, 'HEIGHT'] = imsang['HEIGHT'].apply(lambda x: '00' if x == '0' else x)
imsang.loc[:, 'HEIGHT'] = imsang['HEIGHT'].apply(lambda x: '16' if x == '15' else x)

In [ ]:
class GPUSamplingImsang:
    def __init__(self, imsang_df, imsang_raster, attribute_list=['DNST_CD', 'DMCLS_CD', 'HEIGHT'],
                 h_dict=None, dbh_dict=None, cd_dict=None,
                 h_samples=None, dbh_samples=None, cd_samples=None):
        self.imsang_df = imsang_df.copy()
        self.imsang_raster = imsang_raster
        self.attribute_list = attribute_list

        # 코드별 구간(하한, 상한) 정의
        if h_dict is None:
            self.h_dict = {f"{i*2:02d}": [i*2 - 1, i*2 + 1] for i in range(1, 21)}
            self.h_dict['00'] = [0, 1]
        else:
            self.h_dict = h_dict

        if dbh_dict is None:
            self.dbh_dict = {'0': [0, 6], '1': [6, 18], '2': [18, 30], '3': [30, 107]}
        else:
            self.dbh_dict = dbh_dict

        if cd_dict is None:
            self.cd_dict = {'A': [0.0, 0.50], 'B': [0.50, 0.70], 'C': [0.70, 1.0]}
        else:
            self.cd_dict = cd_dict

        # 샘플 풀 원본
        self.h_samples = cp.asarray(h_samples) if h_samples is not None else None
        self.dbh_samples = cp.asarray(dbh_samples) if dbh_samples is not None else None
        self.cd_samples = cp.asarray(cd_samples) if cd_samples is not None else None

        # 내부 옵션
        self.patch_size = 512
        # self.min_pool_len = 20    # 코드별 샘플풀 최소 길이
        self.h_cur_up_min = 40    # 상한선 최소값
        self.dbh_cur_up_min = 110
        self.cd_cur_up_min = 1.0
        # self.h_expand = 1.0        # 부족 시 확장 폭
        # self.dbh_expand = 6.0
        # self.cd_expand = 0.05
        self.max_expand_attempts = 5

        # 코드 :정수 매핑 (문자열 키를 정수로)
        self.h_code_to_int = {k: i for i, k in enumerate(sorted(self.h_dict.keys()))}
        self.dbh_code_to_int = {k: i for i, k in enumerate(sorted(self.dbh_dict.keys()))}
        self.cd_code_to_int = {k: i for i, k in enumerate(sorted(self.cd_dict.keys()))}

        # 정수 :코드 매칭
        self.h_int_to_code = {v: k for k, v in self.h_code_to_int.items()}
        self.dbh_int_to_code = {v: k for k, v in self.dbh_code_to_int.items()}
        self.cd_int_to_code = {v: k for k, v in self.cd_code_to_int.items()}

        # 사전 샘플풀 (코드 int -> cp.ndarray[float])
        self.h_pool_int = None
        self.dbh_pool_int = None
        self.cd_pool_int = None

        # ID(=df.index 값) -> 정수 코드 인덱스 (GPU 배열)
        self.H_CODE = None
        self.DBH_CODE = None
        self.CD_CODE = None

    def write_log(self, log_content, initialize_log=False):
        """
        로그 파일 생성 및 작성 함수
        """
        curr_dir = os.getcwd()
        if initialize_log:
            mode = 'w'
        else:
            mode = 'a'
        with open(os.path.join(curr_dir, 'log_sampling.txt'), mode, encoding='utf-8') as f:
            f.write(str(log_content) + '\n')

    # ===== 샘플풀 구성 유틸 =====
    def _build_one_pool_dict(self, samples_cp: cp.ndarray, code_bounds: dict, cur_up_min:float,name: str) -> dict: # self.expand
        """
        특성의 각 코드(type: str)별 샘플풀(cp.ndarray(float)) 생성 함수.
        좌우 범위를 expand하여 각 풀의 길이를 min_pool_len까지 확보 시도.
        그래도 부족하면 linspace로 샘플 형성(분포 스무딩 목적).
        Return 값은 dictionary 형태.
        """
        out = {}
        for code, (lo, up) in code_bounds.items():
            pool = samples_cp[(samples_cp > lo) & (samples_cp <= up)]
            attempts = 0
            cur_lo, cur_up = float(lo), float(up)
            """
            while pool.size < self.min_pool_len and attempts < self.max_expand_attempts:
                cur_lo = max(0.0, cur_lo - expand_step)
                cur_up = min(cur_up_min, cur_up + expand_step)
                pool = samples_cp[(samples_cp > cur_lo) & (samples_cp <= cur_up)]
                attempts += 1
            """
            if pool.size == 0: # pool.size < self.min_pool_len:
                # 평균값으로
                avg = (cur_lo + cur_up) / 2
                pool = cp.array([avg])
                # lin = cp.full(cur_lo, cur_up, num=self.min_pool_len, dtype=cp.float32)
                self.write_log(f"[POOL-{name}] code={code} 부족, 평균값으로 대체: {avg}")
            out[code] = pool.astype(cp.float32, copy=False)
        return out

    def _convert_pool_to_indexed_arrays(self, pool_by_code: dict, code_to_int: dict):
        """
        code(str) -> pool(cp.ndarray) 를
        code_int(index) -> pool(cp.ndarray) 형태의 리스트(또는 dict)로 바꾼다.
        """
        max_idx = max(code_to_int.values())
        arr = [None] * (max_idx + 1) # max_idx + 1 == len(code_to_int)
        for code, pool in pool_by_code.items():
            arr[code_to_int[code]] = pool
        # 빈 칸이 없도록 안전장치
        for i, p in enumerate(arr):
            if p is None:
                arr[i] = cp.linspace(0, 1, num=self.min_pool_len, dtype=cp.float32)
        return arr # shape(len(code_to_int), 2)의 리스트

    def _prepare_pools(self):
        """샘플풀을 한 번만 준비"""
        if self.h_samples is None or self.dbh_samples is None or self.cd_samples is None:
            raise ValueError("h_samples, dbh_samples, cd_samples를 모두 제공해야 합니다.")
        
        # sample 코드별 pool 생성
        h_pool_by_code = self._build_one_pool_dict(self.h_samples, self.h_dict, self.h_cur_up_min,'H')
        dbh_pool_by_code = self._build_one_pool_dict(self.dbh_samples, self.dbh_dict, self.dbh_cur_up_min, 'DBH')
        cd_pool_by_code = self._build_one_pool_dict(self.cd_samples, self.cd_dict, self.cd_cur_up_min, 'CD')
        
        # 코드별 pool을 list 형태로 변환
        self.h_pool_int = self._convert_pool_to_indexed_arrays(h_pool_by_code, self.h_code_to_int)
        self.dbh_pool_int = self._convert_pool_to_indexed_arrays(dbh_pool_by_code, self.dbh_code_to_int)
        self.cd_pool_int = self._convert_pool_to_indexed_arrays(cd_pool_by_code, self.cd_code_to_int)

    # ===== ID → 코드 인덱스 매핑 =====
    def _prepare_id_code_maps(self, df_filtered):
        """
        df_filtered.index (정수, 래스터 픽셀값과 일치)에 해당하는 코드의 pool을 GPU에 준비.
        """
        if not np.issubdtype(df_filtered['ID'].dtype, np.integer): # np.issubdtype(): array의 dtype을 확인하는 함수
            raise ValueError("imsang_df의 index가 정수여야 합니다. (래스터 픽셀값과 일치)")

        max_id = int(df_filtered['ID'].max())
        H_CODE = np.full(max_id + 1, -1, dtype=np.int32) # max_id + 1 == len(df_filtered['ID']), "fill_value": -1
        DBH_CODE = np.full_like(H_CODE, -1, dtype=np.int32) # 어떠한 배열(H_CODE)과 똑같은 shape의 배열을 -1로 채워 생성 
        CD_CODE = np.full_like(H_CODE, -1, dtype=np.int32)

        # 문자열 코드 → 정수 코드로 변환
        # attribute_list = ['DNST_CD', 'DMCLS_CD', 'HEIGHT'] 라는 전제 사용
        # DNST_CD -> CD, DMCLS_CD -> DBH?, HEIGHT -> H
        for rid, cd_str, dbh_str, h_str in df_filtered[['ID','DNST_CD','DMCLS_CD','HEIGHT']].itertuples(index=False, name=None):
            try:
                # 각 string 코드에 해당하는 integer 코드를 미리 생성해둔 array에 저장
                H_CODE[rid] = self.h_code_to_int[h_str]
                DBH_CODE[rid] = self.dbh_code_to_int[dbh_str]
                CD_CODE[rid] = self.cd_code_to_int[cd_str]
            except Exception as e:
                # 매핑 실패 시 -1 유지
                self.write_log(f"[IDMAP] rid={rid} 매핑 실패: {e}")

        # 존재하는 code들의 array를 GPU로 올리기
        self.H_CODE = cp.asarray(H_CODE)
        self.DBH_CODE = cp.asarray(DBH_CODE)
        self.CD_CODE = cp.asarray(CD_CODE)

    def run_sampling(self, result_dir=None, patch_size=512):
        os.makedirs(result_dir, exist_ok=True)
        self.patch_size = patch_size

        # === 임상도 전처리 ===
        df = self.imsang_df.dropna(subset=self.attribute_list)
        # 공백 문자열 제거
        df = df[(df[self.attribute_list] != ' ').all(axis=1)].copy()

        # ====샘플풀/ID 맵 준비 (한 번만) ====
        self._prepare_pools()
        self._prepare_id_code_maps(df)

        # ==== Open the reference raster ====
        with rasterio.open(self.imsang_raster) as imsang_ras:
            ref_height, ref_width = imsang_ras.height, imsang_ras.width
            imsang_nodata = imsang_ras.nodata
            profile = imsang_ras.profile.copy()
            block_height, block_width = profile.get('blockysize', 512), profile.get('blockxsize', 512) # blocksize가 없을 경우, 512로 대체
            block_cnt = int(np.ceil(ref_height / block_height) * np.ceil(ref_width/block_width))
            # float타입의 raster에서 NaN을 nodata로 사용할 수 없음!
            new_nodata = imsang_nodata if imsang_nodata is not None else -9999.0
            
            # 대용량(4GB 초과) 레스터 설정으로 profile 갱신
            profile.update(driver='GTiff', dtype='float32', count=1, nodata=new_nodata,
                          tiled=True, blockxsize=block_width, blockysize=block_height,
                            compress="ZSTD", predictor=3, bigtiff='YES')
                            # tiling & compress = ['ZSTD', 'DEFLATE', 'LZW'] → 용량 감소, 안정적 I/O
                            # predictor: float 예측자(압축효율 향상)
                            # bigtiff: 용량 4GB 초과 허용

            # 블록 크기를 소스와 맞추기 (I/O 효율 향상) (드라이버가 제공하지 않으면 무시됨)
            if 'blockxsize' in profile and 'blockysize' in profile: # blocysize, blockxsize: 래스터 파일 내부에서 정사각형 블록 형태로 파일을 쪼개서 저장할 때의 행, 열 길이
                pass

            # ==== 출력 파일을 dictionray 형태로 준비 ====
            out_paths = {
                attr: os.path.join(result_dir, f"{attr}_sampled.tif")
                for attr in self.attribute_list
            }
            dst_files = {
                attr: rasterio.open(out_paths[attr], 'w', **profile)
                for attr in self.attribute_list
            }

            try:
                # === 블록 단위 처리 ===
                for ji, win in tqdm(imsang_ras.block_windows(1), desc="Blocks", total=block_cnt):
                    # .block_windows(1): Band1의 window index와 window 정보 읽어오기 / 예시: (0, 0) Window(col_off=0,   row_off=0,   width=256, height=256)
                    # block_windows()는 'generator', 재사용을 위해 list로 형 변환 but 메모리 많이 사용
                    
                    # 입력 블록 읽기 → GPU
                    ims_block = cp.asarray(imsang_ras.read(1, window=win)) # window에 해당되는 block 읽어오기

                    # 결과를 저장할 array 생성
                    out_cd = cp.full(ims_block.shape, cp.nan, dtype=cp.float32)
                    out_dbh = cp.full_like(out_cd, cp.nan)
                    out_h = cp.full_like(out_cd, cp.nan)

                    valid = ims_block != imsang_nodata
                    if not bool(valid.any()): # "어떤한 값도 유효한 것이 아니라면 == 즉, 모두 nodata이면, np.nan으로 값 채우고 다음 iteration으로 이동(continue)
                        for i_attr, attr in enumerate(self.attribute_list):
                            dst_files[attr].write(out_h.get() if attr == 'HEIGHT' else # .get(): gpu에서 cpu로 값을 불러오는 함수 (cp to np array)
                                                  out_dbh.get() if attr == 'DMCLS_CD' else
                                                  out_cd.get(), 1, window=win)
                        continue

                    # 블록 내 실제 등장 id만 선택
                    uids = cp.unique(ims_block[valid])

                    # uid 단위로 한번에 채우기 (uids 수가 보통 블록 픽셀 수보다 훨씬 적음)
                    for uid in uids.tolist():
                        raster_id = int(uid)           # original ID in raster (1-based)
                        df_id = raster_id - 1          # match df_filtered['ID'] indexing (0-based)

                        mask = (ims_block == raster_id)  # mask uses raster's value
                        n = int(mask.sum())
                        if n == 0:
                            continue

                        if df_id < 0 or df_id >= self.H_CODE.size:
                            continue

                        hci   = int(self.H_CODE[df_id])
                        dbhci = int(self.DBH_CODE[df_id])
                        cdci  = int(self.CD_CODE[df_id])
                        if hci < 0 or dbhci < 0 or cdci < 0:
                            continue

                        hp = self.h_pool_int[hci]
                        dp = self.dbh_pool_int[dbhci]
                        cp_ = self.cd_pool_int[cdci]

                        h_s   = hp[cp.random.randint(0, hp.size, size=n)]
                        dbh_s = dp[cp.random.randint(0, dp.size, size=n)]
                        cd_s  = cp_[cp.random.randint(0, cp_.size, size=n)]

                        out_h[mask]   = h_s
                        out_dbh[mask] = dbh_s
                        out_cd[mask]  = cd_s

                    # 속성별 파일에 raster값 저장 (raster로 저장하지 않고 바로 파일로 저장하여 메모리 비용 감축)
                    dst_files['HEIGHT'].write(out_h.get(), 1, window=win)
                    dst_files['DMCLS_CD'].write(out_dbh.get(), 1, window=win)
                    dst_files['DNST_CD'].write(out_cd.get(), 1, window=win)

            # 예외 발생하더라도 항상 실행
            finally:
                for f in dst_files.values():
                    f.close()


In [ ]:
min(dbh_samples), max(h_samples), max(cd_samples)

In [ ]:
result_dir = r"F:\CBH\result"
raster_file = r"H:\CBH\imsang_Raster2.tif"
si = GPUSamplingImsang(imsang, imsang_raster = raster_file,
                       h_samples=h_samples, dbh_samples=dbh_samples, cd_samples=cd_samples)
si.run_sampling(result_dir=result_dir, patch_size=256)

In [ ]:
# 단위 변환
cm_to_inch = 0.393701
m_to_ft = 3.28084

In [ ]:
visualize_multiple_distribution(sampled_data[:, 2:].T, num_of_col)

# Run the Model

### helpers

In [ ]:
import sklearn, xgboost as xgb
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
# ===== Add to utility =====
# XGBoost 모델 유효성 검사 함수
def check_model_validity(ml_model):
    print("sklearn =", sklearn.__version__)
    print("xgboost =", xgb.__version__)
    print("model type =", type(ml_model))
    
    # 파이프라인 단계 확인
    assert isinstance(ml_model, Pipeline), "Saved object is not a sklearn Pipeline." # isinstnace: type 일치여부를 검토하는 함수
    print("steps =", list(ml_model.named_steps.keys()))
    
    # 전처리 스텝 찾아서 fitted 여부 확인
    pre_names = [k for k,v in ml_model.named_steps.items() if isinstance(v, ColumnTransformer)]
    assert len(pre_names)==1, f"Expected 1 ColumnTransformer, found {pre_names}"
    pre = ml_model.named_steps[pre_names[0]]
    print("preprocessor fitted? ->", hasattr(pre, "transformers_"))
    
    # (있다면) 학습 당시 기대한 컬럼 확인
    if hasattr(pre, "feature_names_in_"):
        print("expects columns =", list(pre.feature_names_in_))


In [ ]:
from datetime import datetime

def write_log(log_content, log_name, block_num, initialize=False):
    """
    로그 파일 생성 및 작성 함수
    log_name: 로그 파일 이름의 suffix
    block_num: 현재 처리 중인 raster block의 순번
    initialize: 강제로 log 파일 초기화하여 작성
    """
    if (block_num == 0) or (initialize==True): open_mode = 'w'
    else: open_mode = "a"
 
    now = datetime.now()
    formatted = now.strftime(r"%Y/%m/%d %H:%M:%S")
    log_content = formatted + "\\t" + log_content

    curr_dir = os.getcwd()

    with open(os.path.join(curr_dir, f'log_{log_name}.txt'), open_mode, encoding='utf-8') as f:
        f.write(str(log_content) + '\n')

### data & model preparation

In [ ]:
model_dir = r"D:\ForestFire\CBH\result\Baseline3\model"
data_dir = r"D:\ForestFire\CBH\data"
result_dir = r"F:\CBH\result"
params_file = r"NFI6+7_train_combined_HM변형-try1.0.csv"

In [ ]:
# ===== WP-B-3: load the NEW plot-disjoint hybrid bundle (replaces the leaky HM-variant1.0 pkl/params) =====
# The bundle is a dict (NOT a sklearn Pipeline), built by src/wp0_plot_disjoint_retrain.py on
# plot-disjoint (GroupShuffleSplit by SampleID) data. It carries everything needed to reproduce the
# map-time prediction without re-fitting: the trained XGB model, the train-fit StandardScaler, the
# per-species baseline recipe (params_by_sid), the sorted label-encoder classes, and the feature schema.
import joblib
BUNDLE_PATH = os.path.join(
    r"D:\ForestFire\CBH\result\Baseline3", "HM변형-try2.0-plotdisjoint", "model",
    "hybrid_plotdisjoint_SEED100.joblib")
bundle = joblib.load(BUNDLE_PATH)

XGB_MODEL      = bundle["xgb_model"]                 # XGBRegressor (10 features, in FEATURE_COLS order)
SCALER         = bundle["scaler"]                    # StandardScaler fit on TRAIN NUMERIC_COLS (9 cols)
PARAMS_BY_SID  = bundle["params_by_sid"]             # {int(SID): ('fit', np.ndarray(4)) | ('mean', float)}
FEATURE_COLS   = list(bundle["FEATURE_COLS"])        # exact 10-col order the XGB model expects
NUMERIC_COLS   = list(bundle["NUMERIC_COLS"])        # the 9 cols the scaler transforms (= FEATURE_COLS - SID_ENC)
LE_CLASSES     = np.asarray(bundle["label_encoder_classes"], dtype=np.int64)  # 39 sorted SIDs
BUNDLE_SEED    = bundle["SEED"]                      # 100 (provenance; record in map metadata/caption)
assert (np.diff(LE_CLASSES) > 0).all(), "label_encoder_classes must be sorted for searchsorted"
print(f"Loaded plot-disjoint bundle: {len(LE_CLASSES)} SIDs, SEED={BUNDLE_SEED}, mode={bundle['mode']}")

# --- baseline func4 (Hasenauer & Monserud 1996), reproduced EXACTLY as in wp0_plot_disjoint_retrain.py ---
# In that script X is built as df[["DBH(inch)","H(ft)"]].values.T, so X row0=DBH(inch), row1=H(ft).
# The internal names H,D are mislabeled but the COLUMN ORDER is [DBH(inch), H(ft)]; replicate that order.
def func4(X, a1, a2, a3, b):
    H, D = X                                          # NOTE: column order is [DBH(inch), H(ft)] (as in retrain script)
    H_log, D_log = np.log1p(H), np.log1p(D)
    z = (a1 * H_log / D_log) + (a2 * H_log) + (a3 * D_log ** 2) + b
    return 1.0 / (1.0 + np.exp(-z))

# --- Species substitution (KOFTR_GROU raster code -> in-bundle SID), per decision 2026-06-22 ---
# Keep ONLY 60->32 and 63->32 (codes absent from the 39-class bundle but present in NFI's alternative
# model). All other raster codes are either already in-bundle (predicted natively) or NOT in the bundle
# (e.g. 77 mixed forest, 78/81-94 non-target) -> MASKED downstream (no substitution, no fabrication).
# NB: the OLD notebook also remapped 19,20,21,65,67 -> {11,32}; those SIDs ARE in the new bundle's
# classes, so remapping them is wrong here. Use only the two NFI substitutions.
SUBSTITUTE_MAP = {60: 32, 63: 32}                     # raster KOFTR_GROU code -> bundle SID
map_keys = cp.asarray(np.asarray(sorted(SUBSTITUTE_MAP.keys()), dtype=np.int64))
map_vals = cp.asarray(np.asarray([SUBSTITUTE_MAP[k] for k in sorted(SUBSTITUTE_MAP.keys())], dtype=np.int64))

# --- GPU lookup tables keyed by the SORTED bundle SIDs (drive both SID_ENC and CR_pred at map time) ---
# SID_ENC = position of the (substituted) SID in LE_CLASSES (classes are sorted, so searchsorted == the
# LabelEncoder integer). A SID not found here is OUT OF BUNDLE and must be masked (incl. 77).
bundle_keys_sorted = cp.asarray(LE_CLASSES)          # (39,) sorted int64 SIDs
# Per-SID baseline recipe -> dense arrays aligned to LE_CLASSES:
#   params row (a1,a2,a3,b) for 'fit' SIDs; is_fit flag; constant CR for 'mean' SIDs.
_params = np.zeros((len(LE_CLASSES), 4), dtype=np.float64)
_is_fit = np.zeros(len(LE_CLASSES), dtype=bool)
_mean_cr = np.zeros(len(LE_CLASSES), dtype=np.float64)
for _i, _sid in enumerate(LE_CLASSES.tolist()):
    _kind, _val = PARAMS_BY_SID[int(_sid)]
    if _kind == "fit":
        _params[_i] = np.asarray(_val, dtype=np.float64); _is_fit[_i] = True
    else:                                             # ('mean', float): constant CR_pred for this species
        _mean_cr[_i] = float(_val)
bundle_params_sorted = cp.asarray(_params)           # (39,4) func4 params per SID (valid where is_fit)
bundle_is_fit        = cp.asarray(_is_fit)           # (39,) True->use func4, False->use constant mean
bundle_mean_cr       = cp.asarray(_mean_cr)          # (39,) constant CR_pred for 'mean' SIDs
print(f"  baseline recipes: {int(_is_fit.sum())} fit / {int((~_is_fit).sum())} mean")


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin

class LabelEncoderWrapper(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.encoders = {}

    def fit(self, X, y=None):
        for col in X.columns:
            le = LabelEncoder()
            le.fit(X[col])
            self.encoders[col] = le
        return self

    def transform(self, X):
        X_encoded = X.copy()
        for col in X.columns:
            X_encoded[col] = self.encoders[col].transform(X[col])
        return X_encoded

    def inverse_transform(self, X):
        X_decoded = X.copy()
        for col in X.columns:
            X_decoded[col] = self.encoders[col].inverse_transform(X[col])
        return X_decoded

In [ ]:
# ===== WP-B-3: this cell is SUPERSEDED and intentionally left as a no-op =====
# It used to load the LEAKY pipeline 'HM-variant1.0-XGBoostGlobal-pipeline.pkl' and defined an older
# hybrid_model() that called ml_model.predict(df_input) on a Pipeline. The deployed prediction path is
# now CELL 31 (bundle load) + the prediction cell below, which use the plot-disjoint bundle. Do not
# re-introduce the pkl/pipeline here. (Left in place to preserve cell numbering / provenance.)
pass


In [ ]:
check_model_validity(ml_model)
ml_model.named_steps.values()

In [ ]:
# ===== Environment variables =====
raster_dir = r"F:\CBH"
block_size = 512
# unit conversions
cm_to_inch = 0.393701
m_to_ft = 3.28084

rasters = glob(os.path.join(raster_dir, '*.tif'))
dem_file    = next((f for f in rasters if 'DEM' in f), None)
# BUG FIX 2 (WP-B-3): the slope predicate was 'DEM' (byte-identical to dem_file), so DEM elevation was
# fed into Slope(tan) and the real Slope raster was never opened. Select the Slope raster instead.
# On F:\CBH the only Slope raster is 'Slope_rad_5179.tif' -> it is in RADIANS (see prediction cell).
slope_file  = next((f for f in rasters if 'Slope' in f), None)
aspect_file = next((f for f in rasters if 'Aspect' in f), None)
height_file = next((f for f in rasters if 'HEIGHT' in f), None)
dbh_file    = next((f for f in rasters if 'DMCLS_CD' in f), None)
cd_file     = next((f for f in rasters if 'DNST_CD' in f), None)
# species raster on disk is KOFTR_INT.tif (int8 KOFTR_GROU codes), not a 'KOFTR_GROU'-named file.
species_file = next((f for f in rasters if 'KOFTR' in f), None)

assert dem_file != slope_file, f"dem_file and slope_file must differ: {dem_file} vs {slope_file}"

raster_paths = {
    'dem' : dem_file, 'slope' : slope_file, 'aspect' : aspect_file,
    'height' : height_file, 'dbh' : dbh_file, 'density' : cd_file,
    'species' : species_file
    }

for rp in raster_paths.items():
    if rp[1] == None:
        raise FileNotFoundError(f"Raster of {rp[0]} doesn't exist in '{raster_dir}'")
print('dem_file   =', os.path.basename(dem_file))
print('slope_file =', os.path.basename(slope_file))
print('species    =', os.path.basename(species_file))


In [ ]:
"""
gdb_dir = r"H:\CBH\Imsang_merge.gdb"
if fiona.listlayers(gdb_dir): # gdb에 레이어가 존재한다면 첫번째 레이어 읽어오기
    layer = fiona.listlayers(gdb_dir)
    imsang = gpd.read_file(os.path.join(gdb_dir), layer=layer[0])
## only once
imsang = imsang.reset_index()
imsang.loc[:, 'index'] = imsang.loc[:, 'index'] + 1

imsang['KOFTR_GROU'] = imsang['KOFTR_GROU'].astype('int')
index_species_dict = imsang.set_index('index')['KOFTR_GROU'].to_dict() # {row.index : row.KOFTR_GROU for row in imsang[['index', 'KOFTR_GROU']].itertuples(index=False)}
"""

In [ ]:
from rasterio.features import rasteriz # submodule: should be imported maually
# ===== Rasterize Feature =====
def rasterize_feature(ref_raster=None, ftype_gdb=None, out_raster=None, resolution=5.0, 
              ref_nodata = -9999.0, all_touched=True, field=None):
    """Rasterize Feature"""
    # ===== read files =====
    # raster file
    print("Reading Files...")
    if ref_raster is not None:
        with rasterio.open(ref_raster) as ref:
            ref_crs = ref.crs
            ref_nodata = ref.nodata if ref.nodata else -9999.
            ref_transform = ref.transform
            w, h = ref.width, ref.height
            resolution = ref.res
    else:
        if isinstance(resolution, (int, float)):
            xres = yres = float(resolution)
        else:
            xres, yres = float(resolution[0]), float(resolution[1])
        
        minx, miny, maxx, maxy, vecs.total_bounds
        ref_crs = vecs.crs
        w = int(math.ceil((maxx - minx) / xres))
        h = int(math.ceil((maxy - miny) / yres))
        ref_transform = rasterio.transform.from_origin(minx, maxy, xres, yres)
        
    # feature file
    if isinstance(ftype_gdb, str):
        vecs = gpd.read_file(ftype_gdb)
    else:
        if fiona.listlayers(gdb): # gdb에 레이어가 존재한다면 첫번째 레이어 읽어오기
            layer = fiona.listlayers(gdb)
            gdf = gpd.read_file(gdb, layer=layer[0])
    # feature validity check
    if vecs.empty:
        raise ValueError(f"Input features are empty!")
    if field not in vecs.columns:
        raise ValueError(f"{field} not found in the input vectors.")
    if vecs.crs is None:
        raise ValueError("Input vectors have no CRS!")
    out_dtype = vecs[field].dtype
    if out_dtype == "O":
        out_dtype = "int32"
        ref_nodata = int(ref_nodata) if (ref_nodata is not None) else -9999
    if out_dtype in ["float32", "float64"]:
        val = lambda v: np.nan if (v is None or (isinstance(v, str) and v.strip() == "")) else float(v)
    elif out_dtype in ["int32", "int64"]:
        val = lambda v: ref_nodata if (v is None or (isinstance(v, str) and v.strip == "")) else int(v)
    # ==== Reproject vectors to raster ====
    if vecs.crs is None:
        print("Reprojection...")
        try:
            with fiona.open(ftype_gdb) as src:
                wkt = getattr(src, "crs_wkt", None)  # may be None if truly missing
            if wkt:
                gdf = gdf.set_crs(CRS.from_wkt(wkt), allow_override=True) # gdf.set_crs("EPSG:5189", allow_override=True) # 
        except Exception:
            pass
    elif vecs.crs != ref_crs:
        print("Reprojection...")
        vecs = vecs.to_crs(ref_crs)
    
    # ==== drop invalid vectors ====
    vecs = vecs[~vecs.geometry.is_empty & vecs.geometry.notna()].copy()
    print("Prepcoessing...")
    # ==== Rasterize =====
    if vecs.empty:
        raster = np.full((h, w), ref_nodata, dtype="float32")
        meta = {
            "transform" : ref_transform,
            "crs" : ref_crs,
            "width" : w,
            "height" : h,
            "nodata" : ref_nodata,
            "dtype" : dtpye,
        }
    else:
        shapes = ((geom,val(value)) for geom, value in zip(vecs.geometry, vecs[field]))
        
        print("Rasterize...")
        raster = rasterize(
                shapes=shapes, 
                out_shape=(h, w),
                transform=ref_transform,
                all_touched = all_touched,
                dtype = out_dtype,
                fill=ref_nodata
        )
        meta = {
            "transform" : ref_transform,
            "crs" : ref_crs,
            "width" : w,
            "height" : h,
            "nodata" : ref_nodata,
            "dtype" : out_dtype,
        }
    
    print("Save the result into the file...")
    with rasterio.open(out_raster, "w", **meta) as dst:
        dst.write(raster, 1)

In [ ]:
# memory error occurs....Need improvements..
rasterize_feature(ref_raster=dem_file, ftype_gdb=r"H:\CBH\Imsang_merge.gdb",
                  out_raster=r"F:\CBH\data\Imsang_KOFTR_GROU.tif",
                  all_touched=True, field="KOFTR_GROU")

### Main

In [ ]:
import warnings
warnings.filterwarnings(
    "error",
    message="invalid value encountered in cast",
    category=RuntimeWarning,
)

In [ ]:
# ===== utilities =====
def create_valid_mask(block, nodata):
# mask helper: returns validity mask given a nodata convention
    if nodata is None or (isinstance(nodata, float) and np.isnan(nodata)):
        return ~cp.isnan(block)
    return block != nodata

STOP_ON_ERROR = True

# ===== model =====
# WP-B-3: use the plot-disjoint bundle (CELL 31) instead of the leaky HM-variant1.0 pipeline pkl.
# XGB_MODEL, SCALER, FEATURE_COLS, NUMERIC_COLS and the GPU lookup tables (bundle_keys_sorted,
# bundle_params_sorted, bundle_is_fit, bundle_mean_cr) are all defined in CELL 31.

# input feature order (FEATURE_COLS): ['H(ft)','DBH(inch)','CD(%)','Elev(hm)','Slope(tan)',
#                                      'Azimuth(rad)','SID_ENC','Lat','Long','CR_pred']
def hybrid_model(df_input):
    """Bundle prediction for a block's valid pixels.

    df_input has the 9 physical features + 'SID_ENC' + 'CR_pred' already built by the caller
    (CR_pred via the bundle's per-species baseline recipe; SID_ENC via searchsorted into the sorted
    label-encoder classes). Here we order to FEATURE_COLS, scale NUMERIC_COLS with the train-fit
    scaler, and predict CR with the bundle's XGBRegressor. Returns predicted CR (true crown ratio).
    """
    X = df_input[FEATURE_COLS].copy()
    X[NUMERIC_COLS] = SCALER.transform(X[NUMERIC_COLS])  # scale with TRAIN stats (SID_ENC left as-is)
    return XGB_MODEL.predict(X)

initialize_log = True
if initialize_log:
    write_log("START PREDICTION", "Prediction", block_num=None, initialize=initialize_log)

# ===== OUTPUT PATHS (WP-B-3) =====
# TODO(WP-B-4): the heavy nationwide run writes NEW tagged rasters; do NOT overwrite the INVALID
# F:\CBH\CBH4.tif / CR4.tif (they carry the old slope+CBH bugs). New tag mirrors the bundle run.
OUT_TAG = "HM변형-try2.0-plotdisjoint_SEED100"   # provenance tag for the corrected maps
OUT_NAMES = {"CR": f"CR_{OUT_TAG}.tif", "CBH": f"CBH_meter_{OUT_TAG}.tif"}  # CR 0-1, CBH in meters

# ===== Save the information from the reference raster =====
with rasterio.open(species_file) as species_ras, \
     rasterio.open(dem_file) as dem_ras, \
     rasterio.open(slope_file) as slp_ras, \
     rasterio.open(aspect_file) as asp_ras, \
     rasterio.open(height_file) as h_ras, \
     rasterio.open(dbh_file) as dbh_ras, \
     rasterio.open(cd_file) as cd_ras:

    ref_height, ref_width = dem_ras.height, dem_ras.width
    ref_transform = dem_ras.transform
    profile = dem_ras.profile.copy()
    b_height, b_width = profile.get('blockysize', block_size), profile.get('blockxsize', block_size)
    block_cnt = int(np.ceil(ref_height / b_height)) * int(np.ceil(ref_width / b_width))
    dem_nodata = dem_ras.nodata if dem_ras.nodata else -9999.
    species_nodata = species_ras.nodata if species_ras.nodata else -9999.
    h_nodata = h_ras.nodata if h_ras.nodata else -9999.
    dbh_nodata = dbh_ras.nodata if dbh_ras.nodata else -9999.
    cd_nodata = cd_ras.nodata if cd_ras.nodata else -9999.

    # Check: all rasters must align (same CRS=EPSG:5179, transform, grid)
    for src in [species_ras, slp_ras, asp_ras, h_ras, dbh_ras, cd_ras]:
        assert src.crs == dem_ras.crs
        assert src.transform == dem_ras.transform
        assert (src.width, src.height) == (dem_ras.width, dem_ras.height)

    # update profile to process the large data
    profile.update(driver = 'GTiff', dtype = 'float32', count=1, nodata=dem_nodata,
                  tiled=True, blockxsize=b_width, blockysize=b_height,
                  compress = 'ZSTD', predictor=3, bigtiff='YES')

    dst_files = {
        attr : rasterio.open(os.path.join(raster_dir, OUT_NAMES[attr]), 'w', **profile)
        for attr in ['CR', 'CBH']
    }
    print("All the files are ready! ->", OUT_NAMES)
    # ===== windowed/blocked prediction =====
    # block layout: height, dbh, cd, elev, slope, azimuth, species(KOFTR_GROU)
    try:
        # iterate by windows; remove the [1820:] slice to do the full extent (WP-B-4)
        for ji, win in tqdm(list(species_ras.block_windows(1))[1820:], desc="Creating CBH...", total = block_cnt):
            try:
                # ===== read blocks to GPU =====
                species_block = cp.asarray(species_ras.read(1, window=win))
                dem_block = cp.asarray(dem_ras.read(1, window=win))
                slp_block = cp.asarray(slp_ras.read(1, window=win))
                asp_block = cp.asarray(asp_ras.read(1, window=win))
                h_block = cp.asarray(h_ras.read(1, window=win))
                dbh_block = cp.asarray(dbh_ras.read(1, window=win))
                cd_block = cp.asarray(cd_ras.read(1, window=win))

                # allocate outputs (GPU)
                out_dtype = cp.float32
                out_cr = cp.full(dem_block.shape, dem_nodata, dtype=out_dtype)
                out_cbh = cp.full(dem_block.shape, dem_nodata, dtype=out_dtype)

                # Validity check & create mask
                dem_ok = create_valid_mask(dem_block, dem_nodata)
                h_ok   = create_valid_mask(h_block,   h_nodata)

                valid_mask = (dem_ok & h_ok)
                if not bool(valid_mask.any()):
                    dst_files['CR'].write(out_cr.get(), 1, window=win)
                    dst_files['CBH'].write(out_cbh.get(),1, window=win)
                    write_log(f"No valid pixel exists in BLOCK{ji}.","Prediction",sum(ji))
                    continue

                # ===== unit conversion (only for valid pixels; matches the bundle's training units) =====
                # H: m -> ft ; DBH: cm -> inch ; Elev: m -> hectometer ; Aspect: deg -> rad
                h_ft_block = cp.where(valid_mask, h_block * m_to_ft, h_block)
                dbh_block = cp.where(valid_mask, dbh_block * cm_to_inch, dbh_block)
                dem_block = cp.where(valid_mask, dem_block / 100.0, dem_block)
                # BUG FIX 2 (units): F:\CBH\Slope_rad_5179.tif is already in RADIANS (range ~0..1.35).
                # The bundle's Slope(tan) feature is tan(slope_angle) with NFI mean ~0.535. So convert
                # radians -> tan directly: cp.tan(slp_block). The old cp.tan(cp.deg2rad(slp_block)) treated
                # radians as degrees and produced values ~1e-2 too small (3 orders of magnitude off).
                slp_block = cp.where(valid_mask, cp.tan(slp_block), slp_block)
                asp_block = cp.where(valid_mask, cp.deg2rad(asp_block), asp_block)

                # ===== Lat/Long per pixel (CPU -> GPU) =====
                h, w = species_block.shape
                rows, cols = np.meshgrid(np.arange(h), np.arange(w), indexing='ij')
                aff = transform(win, ref_transform)
                x, y = xy(aff, rows.ravel(), cols.ravel(), offset='center')
                xs = np.asarray(x).reshape(h, w)
                ys = np.asarray(y).reshape(h, w)
                long_cp = cp.asarray(xs)
                lat_cp = cp.asarray(ys)

                # ===== Species code: apply the 60/63->32 substitution, then resolve SID_ENC =====
                sid_flat = species_block.ravel().astype(cp.int64, copy=False)
                p = cp.searchsorted(map_keys, sid_flat)
                m = (p < map_keys.size) & (map_keys[p] == sid_flat)
                sid_flat_mapped = sid_flat.copy()
                sid_flat_mapped[m] = map_vals[p[m]]

                v_flat = valid_mask.ravel()
                # SID_ENC = position in the sorted bundle classes; match2 is True only for in-bundle SIDs.
                pos2 = cp.searchsorted(bundle_keys_sorted, sid_flat_mapped)
                pos2c = cp.clip(pos2, 0, bundle_keys_sorted.size - 1)
                match2 = (pos2 < bundle_keys_sorted.size) & (bundle_keys_sorted[pos2c] == sid_flat_mapped)
                # COVERAGE POLICY (decision 2026-06-22): out-of-bundle SIDs (incl. KOFTR_GROU=77 mixed
                # forest ~16.84%, and 78/81-94 non-target) are NOT in match2 -> their CR/CBH stay nodata
                # (masked). The masked mixed-forest area MUST be reported in the map caption/limitations
                # (methodology.md S5). Do NOT pass 77 through the encoder; do NOT fabricate a species.
                pred_mask = v_flat & match2

                # ===== Count the valid/predictable pixels =====
                try:
                    valid_cnt = int(valid_mask.sum().get())
                    pred_cnt = int(pred_mask.sum().get())
                except Exception:
                    valid_cnt = int(valid_mask.sum())
                    pred_cnt = int(pred_mask.sum())
                write_log(f"[block {ji}] valid_mask={valid_cnt}, pred_mask={pred_cnt}", "Prediction", sum(ji))

                if not bool(pred_mask.any()):
                    # no in-bundle species in this block -> write nodata and continue
                    dst_files['CR'].write(out_cr.get(), 1, window=win)
                    dst_files['CBH'].write(out_cbh.get(), 1, window=win)
                    write_log(f"Block{ji}: no in-bundle species for any pixels", "Prediction", sum(ji))
                    continue

                # ===== gather per-pixel features for predictable pixels =====
                idx = cp.nonzero(pred_mask)[0]            # flat indices of predictable pixels
                enc_vec = pos2[idx]                       # SID_ENC = searchsorted index (sorted classes)

                # CR_pred from the bundle's per-species baseline recipe (func4 for 'fit', constant for 'mean').
                # func4 column order is [DBH(inch), H(ft)] -> build X accordingly, then clip to [0,1].
                par = cp.take(bundle_params_sorted, enc_vec, axis=0)      # (K,4) a1,a2,a3,b
                isfit = cp.take(bundle_is_fit, enc_vec)                   # (K,) bool
                meancr = cp.take(bundle_mean_cr, enc_vec)                 # (K,) constant CR for 'mean' SIDs

                h_ft_vec = h_ft_block.ravel()[idx]
                h_m_vec = h_block.ravel()[idx]            # true tree height in METERS (for CBH)
                dbh_vec = dbh_block.ravel()[idx]
                cd_vec = cd_block.ravel()[idx]
                dem_vec = dem_block.ravel()[idx]
                slp_vec = slp_block.ravel()[idx]
                asp_vec = asp_block.ravel()[idx]
                lat_vec = lat_cp.ravel()[idx]
                long_vec = long_cp.ravel()[idx]

                # func4 baseline CR_pred (computed on GPU, X=[DBH(inch), H(ft)]):
                Hc = cp.log1p(h_ft_vec); Dc = cp.log1p(dbh_vec)
                Dc = cp.where(Dc == 0, cp.asarray(np.finfo(np.float64).eps), Dc)
                a1, a2, a3, bb = par[:, 0], par[:, 1], par[:, 2], par[:, 3]
                z = (a1 * Hc / Dc) + (a2 * Hc) + (a3 * Dc ** 2) + bb
                cr_pred_fit = 1.0 / (1.0 + cp.exp(-z))
                cr_pred_base = cp.where(isfit, cr_pred_fit, meancr)       # 'mean' SIDs use the constant
                cr_pred_base = cp.clip(cr_pred_base, 0.0, 1.0)            # match retrain-script clip

                # ===== build the feature frame in FEATURE_COLS order =====
                df_input = pd.DataFrame({
                            'H(ft)'       : h_ft_vec.get(),
                            'DBH(inch)'   : dbh_vec.get(),
                            'CD(%)'       : cd_vec.get(),
                            'Elev(hm)'    : dem_vec.get(),
                            'Slope(tan)'  : slp_vec.get(),
                            'Azimuth(rad)': asp_vec.get(),
                            'SID_ENC'     : enc_vec.get().astype(np.int64),
                            'Lat'         : lat_vec.get(),
                            'Long'        : long_vec.get(),
                            'CR_pred'     : cr_pred_base.get(),
                        })

                # ===== predict CR with the bundle (scale NUMERIC_COLS + XGB) =====
                cr_pred = hybrid_model(df_input)         # predicted true crown ratio CR in ~[0,1]

                # ===== write outputs =====
                out_cr.ravel()[idx] = cp.asarray(cr_pred, dtype=out_dtype)
                # BUG FIX 1 (CBH formula): CR is the TRUE crown ratio (H-CBH)/H, so CBH = H*(1-CR).
                # The old line cbh_pred = cr_pred * h_m_vec computed crown LENGTH (H-CBH), not CBH.
                cbh_pred = cp.asarray(1.0 - cr_pred) * h_m_vec
                out_cbh.ravel()[idx] = cp.asarray(cbh_pred, dtype=out_dtype)

                dst_files['CR'].write(out_cr.get(), 1, window=win)
                dst_files['CBH'].write(out_cbh.get(), 1, window=win)

            # =====Exception: write nodata and continue =====
            except Exception as e:
                write_log(f"BLOCK {ji} failed: {type(e).__name__}: {e}", "Prediction", sum(ji))
                traceback.print_exc()
                if STOP_ON_ERROR: raise
                else:
                    try:
                        dst_files['CR'].write(out_cr.get(), 1, window=win)
                        dst_files['CBH'].write(out_cbh.get(), 1, window=win)
                    except Exception:
                        pass
                    continue

    # ===== Finally: always close output files despite exceptions =====
    finally:
        for f in dst_files.values():
            f.close()


In [ ]:
match2.sum()

In [ ]:
def quick_check(path, nodata=None, sample=5):
    with rasterio.open(path) as src:
        arr = src.read(1)
        nd = src.nodata if nodata is None else nodata
        if nd is None or (isinstance(nd, float) and np.isnan(nd)):
            valid = np.isfinite(arr)
        else:
            valid = arr != nd
        print(f"[{path}] shape={arr.shape}, nodata={src.nodata}")
        print("valid count:", int(valid.sum()))
        if valid.any():
            vals = arr[valid]
            print("min/max:", float(vals.min()), float(vals.max()))
            print("samples:", vals.ravel()[:sample])
        else:
            print("No valid pixels.")
            
quick_check(os.path.join(raster_dir, "CR.tif"))
quick_check(os.path.join(raster_dir, "CBH.tif"))


In [ ]:
from rasterio.plot import show
# ===== plot only continuous values =====
def plot_raster(raster_path, band=1, cmap="viridis", title=None):
    with rasterio.open(raster_path) as src:
        data = src.read(band, masked=True)  # masked=True → handles nodata
        fig, ax = plt.subplots(figsize=(8, 6))
        im = show(data, transform=src.transform, ax=ax, cmap=cmap)
        ax.set_title(title if title else f"{raster_path} (band {band})")
        plt.colorbar(ax.images[0], ax=ax, shrink=0.7, label="Value")
        plt.xlabel("X (map units)")
        plt.ylabel("Y (map units)")
        cbar = plt.colorbar(im.get_images()[0], ax=ax, shrink=0.7)
        cbar.set_label('Value')
        plt.tight_layout()
        plt.show()

In [ ]:
plot_raster(r"F:\CBH\CR.tif", title="Crown Ratio")

# Data Augmentation

## Build the Coordinate Rasters

In [ ]:
import geopandas as gpd
from shapely.geometry import Point
import contextily as cxt

In [ ]:
# NFI7 point로 변환
df_nfi7 = pd.read_csv(os.path.join(data_dir, r"NFI7_cleaned2.csv"))
df_nfi7['geometry'] = df_nfi7.apply(lambda row: Point(row['Long'], row['Lat']), axis=1)
gdf_nfi7 = gpd.GeoDataFrame(df_nfi7, geometry='geometry', crs='EPSG:5174')

In [ ]:
# 포인트 시각화
def visualize_point(gdf):
    gdf = gdf.to_crs(epsg=3857) # basemap에 맞춰 좌표 변환
    fig, ax = plt.subplots(figsize=(10, 8))
    gdf.plot(ax=ax, color="blue", markersize=10, edgecolor="gray")
    cxt.add_basemap(ax=ax, source=cxt.providers.OpenStreetMap.Mapnik) # 배경지도 추가 # , zoom=10
    """
    ax.set_xlim(160000, 250000)
    ax.set_ylim(380000, 600000)
    """
    ax.set_title("Points (Republic of Korea)")
    plt.tight_layout()
    plt.show()
    
visualize_point(gdf_nfi7)

In [ ]:
gdf_nfi7 = gdf_nfi7.reset_index()
gdf_nfi7.head(3)

In [ ]:
# point에서 30m Buffer
import random

def generate_random_point(row, cnt):
    points = []
    minx, miny, maxx, maxy = row.bounds
    while len(points) < cnt:
        p = Point(random.uniform(minx, maxx), random.uniform(miny, maxy))
        if row.contains(p): points.append(p)
    return points

imsang_list = [19] # 19, 20, 21, 43, 45, 65, 66, 67, 68, 77
buffer_size = 30
n_points = 5
gdf_nfi7['buffer'] = gdf_nfi7.geometry.apply(lambda geom: geom.buffer(buffer_size)) # distance: raidus of the buffer, resolution: 원모양의 정밀도

random_points = []
parent_ids = []
for imsang_type in tqdm(imsang_list, desc="Extracting random points...", leave=True):
    select_condition = gdf_nfi7['SID'] == imsang_type
    for idx, row in tqdm(gdf_nfi7[select_condition].iterrows(), desc=f"Tree type: {imsang_type}", leave=False):
        points = generate_random_point(row['buffer'], 5)
        random_points.extend(points)
        parent_ids.extend([row['index']] * n_points)
    
gdf_random = gpd.GeoDataFrame({'parent_id' : parent_ids}, geometry=random_points, crs=gdf_nfi7.crs)

In [ ]:
gdf_proj = gdf_nfi7[gdf_nfi7['SID'] == 19].copy().to_crs(epsg=3857)
fig, ax = plt.subplots(figsize=(10, 8))
gdf_proj.plot(ax=ax, color='blue', markersize=100, label='original')
gdf_proj.set_geometry('buffer').plot(ax=ax, facecolor='lightblue', edgecolor='gray', alpha=0.3)
gdf_random.plot(ax=ax, color='red', markersize=50, label='random sampled')

# Force appropriate limits for South Korea EPSG:5174
"""ax.set_xlim(150000, 300000)
ax.set_ylim(350000, 600000)
"""
ax.set_title("Randomly sampled points")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
gdf_proj['buffer'].is_valid, gdf_proj['buffer'].area

# Others

## Build Raster of the NIFS Fuel Model

In [ ]:
df_nfis = pd.read_csv(r"D:\ForestFire\CBH\data\FuelModel-NIFS.csv")
df_nfis.loc[:, ["AGCLS_CD", "DMCLS_CD"]] = df_nfis[["AGCLS_CD", "DMCLS_CD"]].astype('str')
df_alter = pd.read_csv(r"D:\ForestFire\CBH\data\Alternative_Species_Model.csv")
alter_dict = {i : j for i, j in np.array(df_alter[["KOFTR_KR", "MODEL_KR"]])}
alter_dict

In [ ]:
df_nfis['KOFTR_KR'].unique()

In [ ]:
imsang["A_KOFTR_KR"] = [str(i) for i in np.zeros((len(imsang), 1))]
for key in alter_dict.keys():
    condition_imsang = imsang["KOFTR_KR"] == key
    imsang.loc[condition_imsang, "A_KOFTR_KR"] = alter_dict[key]

In [ ]:
df_m = imsang.merge(df_nfis, how="left", left_on=['A_KOFTR_KR', "AGCLS_CD", "DMCLS_CD"], right_on=['KOFTR_KR', "AGCLS_CD", "DMCLS_CD"])

In [ ]:
# nan인 데이터 확인: NIFS 연료모델에 없는 데이터들이라면, 오류 없음.
df_m_nan = df_m.loc[df_m[['H(m)']].isna().any(axis=1), :]
df_m_nan.loc[:, 'AGCLS_CD'] = df_m_nan['AGCLS_CD'].apply(lambda x : -99 if x == " " else x)
df_m_nan.loc[:, 'DMCLS_CD'] = df_m_nan['DMCLS_CD'].apply(lambda x : -99 if x == " " else x)
df_m_nan.loc[:, ['AGCLS_CD', 'DMCLS_CD']] = df_m_nan[['AGCLS_CD', 'DMCLS_CD']].astype('int')
condition1 =((df_m_nan['AGCLS_CD'] > 1) | (df_m_nan['DMCLS_CD'] > 0)) & (~df_m_nan['KOFTR_GROU'].isin([str(i) for i in [77, 78, 81, 82, 83, 91, 92, 93, 94, 95, 99]]))
df_m_nan2 = df_m_nan[condition1]

In [ ]:
valid_cols = ['MAP_LABEL', 'KOFTR_GROU', 'KOFTR_KR_x', 'DNST', 'DBH(cm)_y', 'CBH(m)']
df_m2 = df_m[valid_cols]

In [ ]:
"""
if df_m2.crs is None:
    df_m2.set_crs("EPSG:5179", inplace=True)
print(df_m2.crs)
df_m2.to_file(r"H:\CBH\Imsang_FuelModel.shp")
"""

In [ ]:
print(type(df_m))               # GeoDataFrame이어야 함
print(df_m.geometry.head())     # 각 행이 shapely 객체여야 함

In [ ]:
# save merged dataframe into csv
# geometry를 WKT 문자열로 저장
df_m["geometry"] = df_m["geometry"].apply(lambda geom: geom.wkt)
# csv로 저장
df_m.to_csv(r"H:\CBH\Imsang_FuelModel.csv", encoding='cp949')

# WKT에서 다시 geometry로 변환하기
# df["geometry"] = df["geometry"].apply(wkt.loads)

In [ ]:
# save merged dataframe into csv: only CBH
# geometry를 WKT 문자열로 저장 (WKT(Well-Known Text): standard format for representing geometry as a string)

from shapely import wkt
"""
# 문자열 geometry를 shapely 객체로 변환
# geometry를 WKT 문자열로 저장 (WKT(Well-Known Text): standard format for representing geometry as a string)
if isinstance(df_m2['geometry'].iloc[0], str): # isinstance: 객체의 type 혹은 class 확인
    df_m2["geometry"] = df_m2["geometry"].apply(wkt.loads)
df_m2["geometry"] = df_m2["geometry"].apply(lambda geom: geom.wkt)
"""
# csv로 저장
df_m2.to_csv(r"D:\ForestFire\CBH\data\Imsang_FuelModel_CBH.csv", encoding='cp949')

In [ ]:
# csv to point
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# Load CSV
df_point = pd.read_csv(r"D:\ForestFire\CBH\data\NFI6-7_cleaned2.csv", encoding="cp949")  # or utf-8-sig

# Create geometry column using shapely
geometry = [Point(xy) for xy in zip(df_point["Long"], df_point["Lat"])]

# Create GeoDataFrame
gdf = gpd.GeoDataFrame(df_point, geometry=geometry)

# Set CRS (e.g., EPSG:5174, Korea 2000 / Central Belt)
gdf.set_crs(epsg=5174, inplace=True)

# (Optional) Convert to WGS84 (EPSG:4326)
gdf = gdf.to_crs(epsg=4326)

# Save as shapefile
gdf.to_file(r"D:\ForestFire\CBH\data\NFI_points.shp")

In [ ]:
df_m2.shape